[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fw-ai/cookbook/blob/main/training/case-studies/dpo_serverless/dpo_ultrafeedback_serverless.ipynb)

# DPO from scratch on Fireworks serverless training

Build a complete Direct Preference Optimization loop — the reference model, the loss, the training step —
on `kimi-k3`, with nothing to provision and nothing to tear down.

**Not this notebook if you just want to ship.** Managed DPO jobs ([`../dpo_style/`](../dpo_style/),
`client.dpo_jobs`) do all of this server-side in four SDK calls. Read that one to *use* DPO; read this
one to *understand* it.

**What DPO is.** You show the model **pairs** of answers to the same prompt plus a verdict on which one
a human preferred. No reward function, no reward model, and no sampling during training.

| | needs | generates during training? |
| --- | --- | --- |
| **SFT** | one ideal answer per prompt | no |
| **DPO** | two answers + which is better | **no** |
| **RL (GRPO)** | a prompt + a reward *function* | yes, constantly |

Cheaper and more stable than RL, but its ceiling is your labels — it never explores.

**Why serverless.** Nothing to provision, no GPU quota. That matters here because DPO needs a *second*
frozen model, and serverless gives you one as a snapshot instead of a second GPU job.

**Cost.** Defaults are a real run: ~1 epoch over 2968 pairs, roughly 2.5-3 hours including the
reference pass. Drop `MAX_PAIRS` and `STEPS` for a smoke test.

## 0. Setup

Run this from inside the cookbook repo: the cells below import `training.*` and the shared judge
helpers in `case-studies/common/`. You need `FIREWORKS_API_KEY`.

In [ ]:
# --- Colab only: uncomment this block ------------------------------------
# %pip install -q "fireworks-ai[training]" datasets matplotlib python-dotenv openai
# !git clone -q https://github.com/fw-ai/cookbook.git /content/cookbook
# import os; os.chdir("/content/cookbook/training")   # must chdir: ! cd does not persist
# -------------------------------------------------------------------------

import os, sys, json, time, math, random
from pathlib import Path

# Repo root on sys.path so `training.*` and case-studies/common/* import.
HERE = Path.cwd()
ROOT = next((p for p in [HERE, *HERE.parents] if (p / "training" / "utils").is_dir()), None)
assert ROOT, f"run this notebook from inside the cookbook repo (cwd={HERE})"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "training" / "case-studies" / "common"))

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / "training" / ".env")
except ImportError:
    pass

# Kimi K3 ships a custom tokenizer, so HuggingFace needs explicit permission
# to run the repo's tokenizer code. Set before any get_tokenizer() call.
os.environ.setdefault("HF_TRUST_REMOTE_CODE", "1")

API_KEY = os.environ.get("FIREWORKS_API_KEY")
assert API_KEY, "set FIREWORKS_API_KEY (export FIREWORKS_API_KEY=fw_... or put it in training/.env)"
API_BASE = os.environ.get("FIREWORKS_BASE_URL", "https://api.fireworks.ai")
print("repo root:", ROOT)

In [ ]:
# ----- CONFIG -------------------------------------------------------------
# kimi-k3 is on the serverless TRAINING pool -- a much shorter list than the
# inference catalog (qwen3-8b and qwen3p6-27b are not on it).
BASE_MODEL      = "accounts/fireworks/models/kimi-k3"
TOKENIZER_MODEL = "moonshotai/Kimi-K3"

# PINNED, not auto-resolved. "" gives "kimi_k3", which leaves K3's think block
# OPEN at generation time while training targets close it -- the model then
# reasons past any token budget and sections 9-10 return nothing usable. This
# variant makes the generation prompt and the training target identical.
# Rule: the renderer must match your data. Reasoning traces in your pairs? Use "kimi_k3".
RENDERER_NAME = "kimi_k3_disable_thinking"

LORA_RANK   = 32
MAX_SEQ_LEN = 32768

# STEPS is a CEILING -- EARLY_STOP_CHOSEN_DROP is what ends the run. Aim for
# about one epoch: (MAX_PAIRS - EVAL_PAIRS) / BATCH_PAIRS.
STEPS       = 370
BATCH_PAIRS = 8
LR          = 1e-5         # interacts with LORA_RANK -- see the README
DPO_BETA    = 0.1          # the KL leash (section 4)
EVAL_PAIRS  = 32           # the headline number rests on these; 8 is too few
EVAL_INTERVAL = 20

# Stop when held-out `chosen` falls this many nats below baseline: that is
# likelihood displacement (section 4), and more steps only make it worse.
EARLY_STOP_CHOSEN_DROP = 2.0

# save_state = weights + optimizer state. Portable and resumable, but not
# promotable (that is save_weights_for_sampler). Aligned to EVAL_INTERVAL so
# every step the guard can name is a step you can return to.
DCP_SAVE_INTERVAL = EVAL_INTERVAL
DCP_TIMEOUT_S     = 900.0
RESUME_FROM = None         # "<account>/<run-id>/<checkpoint>" to continue a run

JUDGE_MODEL = "accounts/fireworks/routers/glm-5p2-fast"   # different family: no self-bias
N_WINRATE   = 40

DATA_DIR  = ROOT / "training" / "case-studies" / "dpo_serverless" / "data"
DEMO_DATA = DATA_DIR / "ultrafeedback_preferences.jsonl"
MAX_PAIRS = 3000           # DPO cannot exceed its labels; UltraFeedback has ~63k
DATA_JSONL = None          # point at your own preference JSONL instead

SEED = 0                   # demo reproducibility; comment out for real runs
random.seed(SEED)

## 1. The data: preference pairs

DPO's entire input is **pairs of answers to the same prompt, plus which one won**. No scores, no
rankings, no reward function.

By default we download [UltraFeedback binarized preferences](https://huggingface.co/datasets/argilla/ultrafeedback-binarized-preferences)
and convert it in this notebook — nothing else to run. It lands in a **local JSONL file** (path printed
below) so you can open it and see exactly what DPO consumes, and it's reused on re-runs rather than
re-downloaded.

To train on **your own** data instead, set `DATA_JSONL` in the config above; the validator below checks
it for you.

In [ ]:
from training.utils.data import normalize_preference_row


def download_ultrafeedback(dest, max_pairs, seed=0):
    '''Download UltraFeedback, convert to the chosen/rejected schema, write JSONL.'''
    from datasets import load_dataset

    ds = load_dataset("argilla/ultrafeedback-binarized-preferences", split="train")
    idx = list(range(len(ds)))
    random.Random(seed).shuffle(idx)

    written = 0
    dest.parent.mkdir(parents=True, exist_ok=True)
    with dest.open("w") as f:
        for i in idx:
            if written >= max_pairs:
                break
            ex = ds[i]
            instruction = (ex.get("instruction") or "").strip()
            chosen = (ex.get("chosen_response") or "").strip()
            rejected = (ex.get("rejected_response") or "").strip()
            # Skip empties and no-op pairs: identical responses carry zero signal.
            if not instruction or not chosen or not rejected or chosen == rejected:
                continue
            prompt = [{"role": "user", "content": instruction}]
            line = json.dumps({
                "chosen":   {"messages": [*prompt, {"role": "assistant", "content": chosen}]},
                "rejected": {"messages": [*prompt, {"role": "assistant", "content": rejected}]},
            }, ensure_ascii=False)
            # Escape the line-breaking characters json.dumps leaves raw, so the
            # file is one record per line for ANY reader, not just ours.
            for ch in ("\u2028", "\u2029", "\u0085"):
                line = line.replace(ch, "\\u%04x" % ord(ch))
            f.write(line + "\n")
            written += 1
    return written

def read_jsonl(path):
    return [json.loads(l) for l in path.read_text().split("\n") if l.strip()]

def row_count(path):
    return sum(1 for l in path.read_text().split("\n") if l.strip())


data_path = Path(DATA_JSONL) if DATA_JSONL else DEMO_DATA

if DATA_JSONL:
    print(f"using your data: {data_path}")
else:
    # Re-download when the cached file is SMALLER than MAX_PAIRS. Checking only
    # for existence would silently train on a stale, smaller set after you raise
    # MAX_PAIRS -- a different experiment than the config claims.
    have = row_count(data_path) if data_path.exists() else 0
    if have >= MAX_PAIRS:
        print(f"reusing existing download ({have} rows >= MAX_PAIRS={MAX_PAIRS})")
    else:
        if have:
            print(f"cached file has {have} rows but MAX_PAIRS={MAX_PAIRS} -- re-downloading")
        print("downloading argilla/ultrafeedback-binarized-preferences ...")
        n = download_ultrafeedback(data_path, MAX_PAIRS, seed=SEED)
        print(f"wrote {n} pairs -> {data_path}")

assert data_path.exists(), f"no data at {data_path}"
raw_rows = read_jsonl(data_path)

# MAX_PAIRS is authoritative: a file with extra rows is truncated, not silently
# used in full, so changing MAX_PAIRS always changes the experiment predictably.
if not DATA_JSONL and len(raw_rows) > MAX_PAIRS:
    print(f"file has {len(raw_rows)} rows; using the first MAX_PAIRS={MAX_PAIRS}")
    raw_rows = raw_rows[:MAX_PAIRS]

print(f"{len(raw_rows)} rows loaded ({data_path.stat().st_size / 1024:.0f} KB on disk)")
print(f"\nOpen it yourself:  head -c 800 {data_path}")

### Check the data before you spend anything on it

Three on-disk schemas are accepted (`training/utils/data.py`), all normalized to
`{"chosen": ..., "rejected": ...}`:

| Format | Shape |
| --- | --- |
| chosen / rejected | `{"chosen": {"messages": [...]}, "rejected": {"messages": [...]}}` |
| OpenAI preference | `{"input": {"messages": [...]}, "preferred_output": [...], "non_preferred_output": [...]}` |
| scored samples | `{"samples": [{"messages": [...], "evals": {"score": 1.0}}, {... "score": 0.0}]}` |

The validator checks schema *and* the two failures that produce **no error and a meaningless run**:
**prompt prefixes that differ** between chosen and rejected (the margin then measures the prompt, not
the response) and **identical responses** (zero gradient).

In [ ]:
def validate_preference_rows(rows, show=3):
    '''Validate rows against the three supported DPO schemas.

    Returns (good_rows, report). Checks, ordered by how quietly each breaks a run:
      1. recognized schema (normalize_preference_row accepts it)
      2. both sides carry a non-empty `messages` list
      3. every message has a string `role` and a non-empty string `content`
      4. each side's LAST message is the assistant response being compared
      5. prompt prefixes are IDENTICAL across chosen/rejected  <- the silent killer
      6. the two responses actually differ (identical pair = zero gradient)
    '''
    good, problems = [], []
    fmt = {"chosen/rejected": 0, "input/preferred_output": 0, "samples": 0}

    for i, row in enumerate(rows):
        if not isinstance(row, dict):
            problems.append((i, f"not a JSON object (got {type(row).__name__})"))
            continue

        if "chosen" in row and "rejected" in row:
            fmt["chosen/rejected"] += 1
        elif "preferred_output" in row and "non_preferred_output" in row:
            fmt["input/preferred_output"] += 1
        elif "samples" in row:
            fmt["samples"] += 1

        norm = normalize_preference_row(row)
        if norm is None:
            problems.append((i, "unrecognized schema -- needs chosen/rejected, "
                                "input+preferred_output+non_preferred_output, or scored samples"))
            continue

        sides, failed = {}, None
        for side in ("chosen", "rejected"):
            msgs = (norm.get(side) or {}).get("messages")
            if not isinstance(msgs, list) or not msgs:
                failed = f"{side}: missing or empty 'messages' list"
                break
            for j, m in enumerate(msgs):
                if not isinstance(m, dict):
                    failed = f"{side}[{j}]: message is not an object"
                elif not isinstance(m.get("role"), str) or not m["role"]:
                    failed = f"{side}[{j}]: missing or invalid 'role'"
                elif not isinstance(m.get("content"), str) or not m["content"].strip():
                    failed = f"{side}[{j}]: 'content' must be a non-empty string"
                if failed:
                    break
            if failed:
                break
            if msgs[-1].get("role") != "assistant":
                failed = (f"{side}: last message must be the assistant response "
                          f"(got role={msgs[-1].get('role')!r})")
                break
            sides[side] = msgs
        if failed:
            problems.append((i, failed))
            continue

        prompt_c = [(m["role"], m["content"]) for m in sides["chosen"][:-1]]
        prompt_r = [(m["role"], m["content"]) for m in sides["rejected"][:-1]]
        if prompt_c != prompt_r:
            problems.append((i, "chosen/rejected prompt prefixes DIFFER -- DPO would measure "
                                "the prompt difference, not response quality"))
        elif not prompt_c:
            problems.append((i, "no prompt before the assistant response"))
        elif sides["chosen"][-1]["content"].strip() == sides["rejected"][-1]["content"].strip():
            problems.append((i, "chosen and rejected responses are identical (zero gradient)"))
        else:
            good.append(row)

    report = {"total": len(rows), "valid": len(good), "rejected": len(problems),
              "formats_seen": {k: v for k, v in fmt.items() if v}, "problems": problems}

    print(f"{report['valid']}/{report['total']} rows valid")
    print("formats seen:", report["formats_seen"] or "NONE RECOGNIZED")
    if problems and show:
        print(f"\n{len(problems)} rejected. First {min(show, len(problems))}:")
        for i, msg in problems[:show]:
            print(f"  row {i}: {msg}")
    return good, report


rows, data_report = validate_preference_rows(raw_rows)
assert rows, "no valid preference rows -- fix the problems reported above before training"
assert len(rows) > EVAL_PAIRS, f"need more than EVAL_PAIRS={EVAL_PAIRS} valid rows, got {len(rows)}"

# Hold out the LAST EVAL_PAIRS rows. Fixed membership is what makes the
# before/after numbers comparable; these are never trained on.
train_rows, eval_rows = rows[:-EVAL_PAIRS], rows[-EVAL_PAIRS:]
print(f"\n{len(train_rows)} train / {len(eval_rows)} held out")

In [ ]:
# Look at one pair before touching any math.
ex = normalize_preference_row(train_rows[0])
print("PROMPT:\n ", ex["chosen"]["messages"][0]["content"][:400], "\n")
print("CHOSEN (preferred):\n ", ex["chosen"]["messages"][-1]["content"][:400], "\n")
print("REJECTED:\n ", ex["rejected"]["messages"][-1]["content"][:400])

## 2. Connect to serverless training

One service object hands you both a **training client** (weights that move) and **sampling clients**
(frozen snapshots). No trainer job, no deployment, no lifecycle to manage.

The **renderer** is the chat-template layer: it turns a message list into the exact token sequence this
base model was trained on. We pin it rather than auto-resolving — see the CONFIG comment.

This attaches to a shared pool, so it needs no GPU quota. It can queue for a minute or two when the
pool is busy, and can fail outright if the pool is at capacity — just re-run the cell.

In [ ]:
import tinker
from fireworks.training.sdk import FiretitanServiceClient
from training.renderer import get_text_content
from training.renderer.tokenizer import get_tokenizer
from training.utils.losses import make_batch_dpo_loss_fn
from training.utils.supervised import (
    build_renderer,
    render_preference_pair,
    resolve_renderer_name,
)


def serverless_base_url(base_url: str) -> str:
    root = base_url.rstrip("/")
    if root.endswith("/training/v1/serverless"):
        return root
    if root.endswith("/training/v1"):
        return f"{root}/serverless"
    return f"{root}/training/v1/serverless"


tokenizer = get_tokenizer(TOKENIZER_MODEL)
# RENDERER_NAME = "" means "resolve from the tokenizer". Resolve it explicitly
# too, so the name actually in use is printed rather than an empty string.
resolved_renderer = RENDERER_NAME or resolve_renderer_name(TOKENIZER_MODEL)
renderer = build_renderer(tokenizer, TOKENIZER_MODEL, resolved_renderer)

service = FiretitanServiceClient(api_key=API_KEY, base_url=serverless_base_url(API_BASE))

if RESUME_FROM:
    # Restores weights AND Adam momentum. The reference then anchors at the
    # RESUMED weights, so margins are relative to the resume point. Keep
    # dataset, SEED and BATCH_PAIRS identical.
    print(f"resuming from {RESUME_FROM}")
    training_client = service.create_training_client_from_state_with_optimizer(RESUME_FROM)
else:
    training_client = service.create_lora_training_client(base_model=BASE_MODEL, rank=LORA_RANK)

print(f"attached to pooled trainer for {BASE_MODEL}")
print(f"  LoRA rank {LORA_RANK}, renderer {resolved_renderer}"
      f"{' (auto-resolved)' if not RENDERER_NAME else ' (pinned)'}")
print(f"  training session: {service.training_session_name or service.training_session_id}")

## 3. The reference model

DPO needs a **frozen copy of the model you started from** — a "reference." Its only job is to answer:
*what did you think of this text before training?*

On serverless you get one in two lines. `save_weights_for_sampler` writes the current weights to a
snapshot; because the LoRA adapter is **zero-initialized** (LoRA's `B` matrix starts at zero, so the
adapter is a no-op until the first optimizer step), that snapshot *is* the untouched base model. Take
it **before any optimizer step** and it stays frozen for the whole run.

From here on two copies exist: the training client's weights (moving) and this snapshot (frozen).

One clarification about what this snapshot is *for*. DPO needs the reference's **logprobs**, and we get
those from the training client itself in section 5, before any optimizer step — same engine as the
policy, which matters more than it sounds like it should. This snapshot exists so we can later
**generate** from the untouched base model and read it side by side with the tuned one in sections 9-10.
Scoring and sampling, two different jobs.

In [ ]:
ref_path = training_client.save_weights_for_sampler("dpo-ref").result().path
assert ref_path, "save_weights_for_sampler returned no path"
reference_client = service.create_sampling_client(model_path=ref_path, tokenizer=tokenizer)
print("frozen reference:", ref_path)

## 4. What the reference buys you, and the trap it does *not* prevent

Run this cell — it needs no GPU.

DPO turns each response's raw log-probability into an **implicit reward** measured against the reference:

$$r(y) = \beta\left[\log \pi_{\text{policy}}(y) - \log \pi_{\text{ref}}(y)\right]$$

$$\mathcal{L} = -\log \sigma\big(r(\text{chosen}) - r(\text{rejected})\big)$$

So "reward" means **how much more likely this text became relative to the base model** (in log units,
scaled by β), not absolute probability. The reference is the KL anchor from the RLHF objective that DPO
solves in closed form. That is what makes β a real leash: it bounds how far the policy may drift from
where it started.

`-log σ(·)` is logistic regression on the margin — the Bradley-Terry model of a pairwise preference. It
is minimized by making the chosen response beat the rejected one, and it **saturates**, so a pair the
model already ranks correctly stops contributing gradient.

What the reference does **not** do is force the chosen response to become *more* likely. Look at row (b).

In [ ]:
sigmoid = lambda x: 1 / (1 + math.exp(-x))
REF = {"chosen": -10.0, "rejected": -20.0}   # the frozen reference's view of one pair

scenarios = {
    "start (policy == ref)":      (REF["chosen"], REF["rejected"]),
    "(a) learned it":             (-5.0,   -20.0),   # chosen went UP. what we want.
    "(b) displacement (typical)": (-14.0,  -40.0),   # BOTH went down, rejected further.
    "(c) collapse (extreme)":     (-100.0, -500.0),  # same shape, model destroyed.
}

hdr = f"{'scenario':28s} {'logp_chosen':>11s} {'logp_rej':>9s} {'r_chosen':>9s} {'r_rej':>8s} {'r_margin':>9s} {'loss':>8s}"
print(hdr); print("-" * len(hdr))
for name, (lp_c, lp_r) in scenarios.items():
    r_c = DPO_BETA * (lp_c - REF["chosen"])
    r_r = DPO_BETA * (lp_r - REF["rejected"])
    loss = -math.log(sigmoid(r_c - r_r))
    print(f"{name:28s} {lp_c:11.1f} {lp_r:9.1f} {r_c:+9.2f} {r_r:+8.2f} {r_c - r_r:+9.2f} {loss:8.4f}")

Row **(b)** is the one to sit with: the loss is ~4x better than the start, and the chosen response
became **4x less likely** than the base model made it. The objective is satisfied only because the
rejected response fell further. Row (c) is the same shape taken to its conclusion.

This is **likelihood displacement**: the margin can rise while the chosen response's own likelihood
falls. The reference bounds drift in units of β; it does not forbid this.

**So watch `chosen_reward` and `rejected_reward` separately.** Healthy looks like:

```
chosen_reward   ↑  (or at least flat)
rejected_reward ↓
```

Both falling together means you are degrading the model while the loss curve improves. Remedies: lower
`LR`, raise `DPO_BETA`, or switch to **ORPO** ([`recipes/orpo_loop.py`](../../recipes/orpo_loop.py)),
whose extra `L_SFT(chosen)` term penalizes exactly this. `EARLY_STOP_CHOSEN_DROP` watches it for you.

> **Three things get called "margin."** `r_margin` above is β-scaled. The **trainer's** `margin` is the
> same quantity *without* β (~10x larger at β=0.1; same sign, same diagnostic). Section 7's **held-out
> margin** is a third thing: plain `log π(chosen) − log π(rejected)`, no reference, no β.

## 5. Render the pairs, then score them against the reference

**Render.** `render_preference_pair` tokenizes both sides into **datums** — a tokenized sequence plus a
per-token weight for which tokens count. Prompt tokens get weight 0, response tokens 1: both sides
share the identical prompt, so training on it would be noise.

**Score with the reference.** We compute the frozen model's logprobs **once, up front, through the
training client's own `forward`** — before any `optim_step`, so the weights it sees *are* the step-0
reference weights.

Same engine matters: asking the reference *sampling* client instead works, but the sampling and
training engines return different logprobs for identical weights (different kernels, and for MoE
different expert routing). Measured, that put a **±2-3 noise floor on `margin` per pair** — larger than
the signal — which makes `margin`, `accuracy`, and both rewards unreadable. Scoring with the same engine
that later computes the policy's logprobs makes step 0 agree to exactly 0.0. RL loops track this same
divergence as `kld/mean_k3`; in DPO it lands inside the objective.

Cached by row index and never recomputed — the reference is frozen.

In [ ]:
def render_pair(row):
    norm = normalize_preference_row(row)
    if norm is None:
        raise ValueError("row does not match a supported preference schema")
    pair = render_preference_pair(
        norm["chosen"], norm["rejected"],
        renderer=renderer, tokenizer=tokenizer, max_seq_len=MAX_SEQ_LEN,
    )
    if pair is None:
        raise ValueError("pair failed to render (empty messages or over max_seq_len)")
    return pair


def interleave(pairs):
    '''[chosen_0, rejected_0, chosen_1, rejected_1, ...] -- the layout the loss pairs by index.'''
    return [d for p in pairs for d in (p.chosen_datum, p.rejected_datum)]

Schema validity isn't the whole story: a pair also has to **render and fit**. This is the second half
of the data check, and it needs the tokenizer, which is why it lives here rather than in section 1.
`render_preference_pair` returns `None` when a pair exceeds `MAX_SEQ_LEN`, so we drop those now rather
than crashing mid-training.

In [ ]:
def validate_rendering(rows, label):
    '''Drop pairs that fail to render or exceed MAX_SEQ_LEN; report token stats.'''
    kept, dropped, lengths = [], [], []
    for i, row in enumerate(rows):
        try:
            pair = render_pair(row)
        except ValueError as e:
            dropped.append((i, str(e)))
            continue
        lengths += [pair.chosen_datum.model_input.length,
                    pair.rejected_datum.model_input.length]
        kept.append(row)

    if lengths:
        lengths.sort()
        print(f"{label}: {len(kept)}/{len(rows)} pairs render | "
              f"tokens min {lengths[0]} / median {lengths[len(lengths) // 2]} / max {lengths[-1]} "
              f"(MAX_SEQ_LEN {MAX_SEQ_LEN})")
    else:
        print(f"{label}: 0/{len(rows)} pairs render")
    for i, msg in dropped[:3]:
        print(f"  dropped row {i}: {msg}")
    return kept


train_rows = validate_rendering(train_rows, "train")
eval_rows = validate_rendering(eval_rows, "held out")
assert train_rows and eval_rows, "no renderable pairs left -- raise MAX_SEQ_LEN or check the renderer"

Don't take that on faith — check it on **your** data. This prints the assistant-turn scaffold your
renderer emits and the exact span the DPO loss compares. If the compared span doesn't start at your
response text, your renderer doesn't match your data.

In [ ]:
def show_training_boundary(row):
    pair = render_pair(row)
    rs = pair.response_start
    c = pair.chosen_datum.model_input.to_ints()
    j = pair.rejected_datum.model_input.to_ints()

    print(f"response_start = {rs}   chosen len = {len(c)}   rejected len = {len(j)}")
    print(f"prefixes identical up to response_start: {c[:rs] == j[:rs]}")
    print()
    print("assistant-turn scaffold the renderer emits (the last 12 prompt tokens):")
    print("   ", repr(tokenizer.decode(c[max(0, rs - 12):rs])))
    print()
    print(f"what the DPO loss actually compares (from index rs-1 = {rs - 1}):")
    print("    chosen  :", repr(tokenizer.decode(c[rs - 1:rs + 14])))
    print("    rejected:", repr(tokenizer.decode(j[rs - 1:rs + 14])))


show_training_boundary(train_rows[0])

Now the reference pass. **This must run before the first `optim_step`** — that is what makes these the
step-0 weights. It is the one upfront cost in the notebook; everything after it is cached.

In [ ]:
REF_CACHE = {}


def precompute_reference_logprobs(indexed_rows, pairs_per_forward=None, timeout=900):
    '''Reference logprobs for every row, from the TRAINING client's forward pass.

    Must be called before any optim_step: the weights the forward sees are then
    still the step-0 (zero-init LoRA = base model) reference weights.

    Returns nothing; fills REF_CACHE keyed by dataset row index. The arrays come
    back already in the datum layout the DPO loss slices, so no realignment is
    needed -- unlike a sampling-client compute_logprobs, which prepends a None.
    '''
    # MUST match the loop's batch size: trainer logprobs are batch-size
    # dependent (measured at up to ~2.8 nats on a single token), so a mismatch
    # reintroduces the noise this cell exists to remove.
    pairs_per_forward = pairs_per_forward or BATCH_PAIRS

    todo = [(i, row) for i, row in indexed_rows if i not in REF_CACHE]
    if not todo:
        return
    for start in range(0, len(todo), pairs_per_forward):
        chunk = todo[start:start + pairs_per_forward]
        pairs = [render_pair(row) for _, row in chunk]
        fwd = training_client.forward(interleave(pairs), "cross_entropy").result(timeout=timeout)
        for k, (idx, _) in enumerate(chunk):
            REF_CACHE[idx] = (
                [float(x) for x in fwd.loss_fn_outputs[2 * k]["logprobs"].data],
                [float(x) for x in fwd.loss_fn_outputs[2 * k + 1]["logprobs"].data],
            )
        done = min(start + pairs_per_forward, len(todo))
        print(f"  reference logprobs: {done}/{len(todo)} pairs", end="\r", flush=True)
    print(f"  reference logprobs: {len(todo)}/{len(todo)} pairs cached")


# Fix the training order HERE so we score exactly the rows the loop visits.
# REF_CACHE is keyed by index into train_rows, so train_rows must be FINAL --
# re-running the render-validation cell above would shift every index.
REF_CACHE_ROWS = len(train_rows)

order = list(range(len(train_rows)))
random.Random(SEED).shuffle(order)
visited = [order[i % len(order)] for i in range(STEPS * BATCH_PAIRS)]
needed = list(dict.fromkeys(visited))          # de-duped, order preserved

# Step-driven, unlike recipes/dpo_loop.py (`epochs`). Print the derived
# epoch count rather than make you divide.
epochs = len(visited) / len(train_rows)
print(f"{STEPS} steps x {BATCH_PAIRS} pairs = {len(visited)} pair-visits "
      f"over {len(needed)} distinct rows")
print(f"  = {epochs:.2f} epochs of the {len(train_rows)}-row train pool "
      f"({len(train_rows) - len(needed)} rows unused)"
      if epochs <= 1 else
      f"  = {epochs:.2f} epochs of the {len(train_rows)}-row train pool "
      f"(each pair seen ~{len(visited) / len(needed):.1f}x)")
print(f"  one full epoch would be STEPS = {len(train_rows) // BATCH_PAIRS}")
t0 = time.time()
precompute_reference_logprobs([(i, train_rows[i]) for i in needed])
print(f"done in {time.time() - t0:.1f}s")

Now **prove the reference is what we claimed**: same weights, same engine, so the policy's logprobs at
step 0 must equal the cached reference logprobs *exactly*. Any nonzero value here means either an
optimizer step already ran, or the reference came from a different engine.

In [ ]:
# Compare using a FULL BATCH_PAIRS batch -- the same shape the reference pass
# and the training loop use. Scoring a single pair here would change the batch
# size and disagree by nats even though nothing is wrong.
_idx = list(REF_CACHE)[:BATCH_PAIRS]
_pairs = [render_pair(train_rows[i]) for i in _idx]
_pol = training_client.forward(interleave(_pairs), "cross_entropy").result()

_delta = 0.0
for _k, _i in enumerate(_idx):
    for _side in (0, 1):
        _ref = REF_CACHE[_i][_side]
        _cur = [float(x) for x in _pol.loss_fn_outputs[2 * _k + _side]["logprobs"].data]
        _delta = max([_delta] + [abs(a - b) for a, b in zip(_ref, _cur)])

print(f"max |policy - reference| logprob at step 0: {_delta:.2e}   (must be 0.0)")
print("If this is nonzero, either an optim_step already ran, or the reference")
print("was scored with a different batch size / engine than the policy.")

### Resumable checkpoints

`save_state` writes weights **plus optimizer state**, so a resume is a true continuation rather than a
warm start from cold momentum. It is a different artifact from the sampler snapshots sections 9-11 use —
see the config comment for the split.

The reference it prints is **portable**: unlike a sampler snapshot, it outlives this session and can be
loaded by a fresh kernel.

In [ ]:
saved_checkpoints = []


def dcp_reference(name):
    '''The portable `<account>/<run-id>/<checkpoint>` reference for RESUME_FROM.'''
    session = getattr(service, "training_session_name", None) or ""
    account = session.split("/")[1] if session.startswith("accounts/") else None
    run_id = getattr(training_client, "run_id", None)
    return f"{account}/{run_id}/{name}" if account and run_id else name


def save_dcp(step):
    '''Full trainer state -- what RESUME_FROM consumes. The name must end in a
    zero-padded step number; the resume path parses the step back out of it.
    '''
    name = f"dpo-{step:04d}"
    t0 = time.time()
    training_client.save_state(name).result(timeout=DCP_TIMEOUT_S)
    ref = dcp_reference(name)
    saved_checkpoints.append((step, ref))
    print(f"     checkpoint {name} ({time.time() - t0:.0f}s) -> {ref}")
    return ref

## 6. The training step

Everything above assembles into one function. Note what is **absent** compared to an RL loop: no
sampling, no reward function, no exploration, no group statistics. Just render, score against the
reference, and update.

One API detail: this calls **`forward_backward_custom`**, not `forward_backward`. A built-in loss (like
RL's `"importance_sampling"`) works on per-token arrays inside a single datum. DPO compares *two whole
sequences*, so the loss has to be a function you pass in — closed over the frozen reference arrays.

In [ ]:
adam = tinker.AdamParams(learning_rate=LR, beta1=0.9, beta2=0.95, eps=1e-12, weight_decay=0.0)


def dpo_step(indexed_batch):
    pairs = [(i, render_pair(r)) for i, r in indexed_batch]
    rendered = [p for _, p in pairs]

    loss_fn = make_batch_dpo_loss_fn(
        [REF_CACHE[i][0] for i, _ in pairs],    # reference chosen logprobs (cached at init)
        [REF_CACHE[i][1] for i, _ in pairs],    # reference rejected logprobs
        [p.response_start for p in rendered],   # where each response begins
        DPO_BETA,
    )
    fb = training_client.forward_backward_custom(interleave(rendered), loss_fn).result()
    training_client.optim_step(adam).result()

    m = getattr(fb, "metrics", None) or {}
    return {k: m.get(k) for k in
            ("dpo_loss", "margin", "accuracy", "chosen_reward", "rejected_reward")}

## 7. Held-out margin: the baseline

A plain `forward` on the fixed held-out pairs — no backward, no optimizer. It returns **three** numbers,
because the margin alone is a *gap*: a gap widens identically whether chosen rose or both fell with
rejected falling faster. That is the section 4 blind spot, so we track chosen and rejected separately.

**The baseline value is not the result — the delta is.** Its *sign* is not meaningful either: this sums
logprobs, and longer text sums to a more negative number. On this dataset chosen responses average 277
tokens against rejected's 185, worth roughly 50 nats of margin from length alone. Length cancels in the
delta. (The DPO loss sums the same way, which is why DPO tends to make models more verbose.)

In [ ]:
def held_out_margin(pairs_per_forward=None):
    """Mean held-out (chosen, rejected, margin) summed logprobs.

    All three, not just the margin: a gap widens identically whether chosen rose
    or both fell with rejected falling faster (section 4). Chunked at
    BATCH_PAIRS per forward, same reason as the reference pass.
    """
    pairs_per_forward = pairs_per_forward or BATCH_PAIRS
    pairs = [render_pair(r) for r in eval_rows]
    tot_c = tot_r = 0.0
    for s in range(0, len(pairs), pairs_per_forward):
        chunk = pairs[s:s + pairs_per_forward]
        fwd = training_client.forward(interleave(chunk), "cross_entropy").result()
        for k, pair in enumerate(chunk):
            lp = max(0, pair.response_start - 1)
            tot_c += sum(fwd.loss_fn_outputs[2 * k]["logprobs"].data[lp:])
            tot_r += sum(fwd.loss_fn_outputs[2 * k + 1]["logprobs"].data[lp:])
    n = len(pairs)
    return {"chosen": tot_c / n, "rejected": tot_r / n, "margin": (tot_c - tot_r) / n}


def print_held_out(label, m, base=None):
    if base is None:
        print(f"{label}:  chosen {m['chosen']:+9.3f} | rejected {m['rejected']:+9.3f} "
              f"| margin {m['margin']:+9.3f}")
    else:
        print(f"{label}:  chosen {m['chosen']:+9.3f} ({m['chosen'] - base['chosen']:+7.3f}) "
              f"| rejected {m['rejected']:+9.3f} ({m['rejected'] - base['rejected']:+7.3f}) "
              f"| margin {m['margin']:+9.3f} ({m['margin'] - base['margin']:+7.3f})")


before = held_out_margin()
eval_history = [{"step": 0, **before}]
print_held_out("held-out @ step   0", before)

## 8. Run the loop

`STEPS` is a ceiling; `EARLY_STOP_CHOSEN_DROP` is what usually ends the run.

Two reading notes:

- **`accuracy` and `margin` start at exactly 0.0.** Accuracy counts pairs whose *reference-adjusted*
  margin is strictly `> 0`. At step 0 the policy **is** the reference and both were scored by the same
  engine, so every margin is exactly zero. Nonzero here means the reference was scored by a different
  engine — see section 5.
- Per-step `margin` has a standard deviation around 0.7 purely from batch composition, so read trends
  over ten-plus steps. The held-out lines are what settle it.

> **`chosen_r` is the diagnostic, not the loss.** Divide it by `DPO_BETA` for nats moved against base.
>
> | symptom | reading |
> | --- | --- |
> | loss pinned at `ln 2`, `chosen_r` ±0.03 | not learning — raise `LR` or `LORA_RANK` |
> | `rej_r` falling, `chosen_r` flat or rising | healthy |
> | both falling, `chosen_r` diving | **likelihood displacement** — lower `LR`, raise `DPO_BETA`, or use ORPO |
>
> The trap: a displaced run has the *best* loss curve, the *best* accuracy, and the *largest* margin of
> any configuration. Every headline number says it won. Section 4 is about why.

In [ ]:
# `order` was fixed in section 5 so the reference pass scored exactly these
# rows; re-shuffling here would visit rows with no cached reference logprobs.
assert len(train_rows) == REF_CACHE_ROWS, (
    "train_rows changed after the reference pass -- REF_CACHE indices are stale. "
    "Re-run the reference-pass cell in section 5."
)
history, cursor = [], 0
best_eval = {"step": None, "gain": float("-inf"), "drop": 0.0}
stopped_early = False
fmt = lambda v, s="+.4f": "n/a" if v is None else format(v, s)

print(f"{'step':>4s} {'loss':>8s} {'margin':>9s} {'acc':>7s} {'chosen_r':>9s} {'rej_r':>9s} {'secs':>6s}")
for step in range(STEPS):
    idx = [order[(cursor + i) % len(order)] for i in range(BATCH_PAIRS)]
    cursor += BATCH_PAIRS
    t0 = time.time()
    rec = dpo_step([(i, train_rows[i]) for i in idx])
    rec["step"], rec["wall"] = step, time.time() - t0
    history.append(rec)
    print(f"{step:4d} {fmt(rec['dpo_loss'], '.4f'):>8s} {fmt(rec['margin']):>9s} "
          f"{fmt(rec['accuracy'], '.1%'):>7s} {fmt(rec['chosen_reward']):>9s} "
          f"{fmt(rec['rejected_reward']):>9s} {rec['wall']:6.1f}")

    # Same fixed pairs every time, so this is the curve to trust.
    if EVAL_INTERVAL and (step + 1) % EVAL_INTERVAL == 0 or step == STEPS - 1:
        ev = {"step": step, **held_out_margin()}
        eval_history.append(ev)
        print_held_out(f"     held-out @ step {step:3d}", ev, before)

        if DCP_SAVE_INTERVAL:
            save_dcp(step)

        # Best point = largest margin gain while chosen has not fallen more
        # than half the stop threshold. (Held-out chosen wobbles ~0.1 nats, so
        # requiring drop <= 0 rejects every good step.)
        gain = ev["margin"] - before["margin"]
        drop = before["chosen"] - ev["chosen"]          # positive = chosen fell
        if drop <= (EARLY_STOP_CHOSEN_DROP or 2.0) / 2 and gain > best_eval["gain"]:
            best_eval = {"step": step, "gain": gain, "drop": drop}

        # Early stop on displacement. The margin will keep climbing past this
        # point; the model will keep getting worse at the preferred answers.
        if EARLY_STOP_CHOSEN_DROP and drop > EARLY_STOP_CHOSEN_DROP:
            print()
            print(f"  EARLY STOP at step {step}: held-out chosen fell {drop:.2f} nats "
                  f"(limit {EARLY_STOP_CHOSEN_DROP}).")
            print(f"  This is likelihood displacement -- margin is up {gain:+.2f} but the")
            print(f"  preferred responses became LESS likely. Running longer makes it worse.")
            if best_eval["step"] is not None:
                print(f"  Best point seen: step {best_eval['step']} "
                      f"(margin {best_eval['gain']:+.2f}, chosen {-best_eval['drop']:+.2f}).")
                print(f"  Set STEPS = {best_eval['step'] + 1} to stop there, or lower LR / raise DPO_BETA.")
                _ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
                if _ref:
                    print(f"  Resume from that point with  RESUME_FROM = \"{_ref}\"")
            stopped_early = True
            break

# The last eval entry IS the current state -- recomputing it would waste an
# eval and, on an early stop, label it STEPS-1, a step that never ran.
after = eval_history[-1]
if not stopped_early and after["step"] != STEPS - 1:
    after = {"step": STEPS - 1, **held_out_margin()}
    eval_history.append(after)
    print()
    print_held_out(f"held-out @ step {STEPS - 1:3d}", after, before)

d_c = after["chosen"] - before["chosen"]
d_r = after["rejected"] - before["rejected"]
d_m = after["margin"] - before["margin"]

print()
print(f"held-out margin: {before['margin']:+.4f} -> {after['margin']:+.4f}  (delta {d_m:+.4f})")
if d_m > 0 and d_c < 0:
    share = abs(d_c) / (abs(d_c) + abs(d_r)) if (d_c or d_r) else 0
    print(f"\n  !! LIKELIHOOD DISPLACEMENT: the margin rose {d_m:+.2f} but the CHOSEN")
    print(f"     response became {abs(d_c):.2f} nats LESS likely. The gain came from the")
    print(f"     rejected response collapsing -- chosen is only {share:.0%} of the movement.")
    print(f"     See section 4: lower LR, raise DPO_BETA, or switch to ORPO.")
elif d_m > 0:
    print(f"\n  healthy: margin up {d_m:+.2f} with chosen {d_c:+.2f} (not falling).")
else:
    print(f"\n  margin did not improve ({d_m:+.2f}). See the troubleshooting note above.")

if stopped_early:
    _best_ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
    print(f"\n  NOTE: these weights are from step {after['step']}, which is past the best")
    print(f"        point. The loop cannot rewind -- but the checkpoints can.")
    if _best_ref:
        print(f"        Recover the best weights in a fresh kernel with:")
        print(f"          RESUME_FROM = \"{_best_ref}\"")
    elif saved_checkpoints:
        print(f"        Nearest saved checkpoint: {saved_checkpoints[-1][1]}")
    else:
        print(f"        No checkpoints were saved (DCP_SAVE_INTERVAL = 0), so the only")
        print(f"        way back is a re-run with a smaller STEPS.")
elif best_eval["step"] is not None and best_eval["step"] < STEPS - 1 and d_c < -0.5:
    print(f"\n  NOTE: the best held-out point was step {best_eval['step']} "
          f"(margin {best_eval['gain']:+.2f}, chosen {-best_eval['drop']:+.2f}); "
          f"the run continued past it.")
    _ref = next((r for st, r in saved_checkpoints if st == best_eval["step"]), None)
    if _ref:
        print(f"        these weights are from step {after['step']}. To get the best ones,")
        print(f"        set RESUME_FROM = \"{_ref}\" in a fresh kernel.")

if saved_checkpoints:
    print(f"\nresumable checkpoints ({len(saved_checkpoints)}):")
    for st, ref in saved_checkpoints[-5:]:
        print(f"  step {st:4d}  {ref}")
    if len(saved_checkpoints) > 5:
        print(f"  ... and {len(saved_checkpoints) - 5} earlier")

In [ ]:
import matplotlib.pyplot as plt


def series(key, src=None):
    """Steps and values for one metric, skipping entries that reported None."""
    pts = [(r["step"], r[key]) for r in (src or history) if r.get(key) is not None]
    return [p[0] for p in pts], [p[1] for p in pts]


fig, ax = plt.subplots(1, 4, figsize=(19.5, 3.8))

# --- 1. loss -------------------------------------------------------------
ax[0].plot(*series("dpo_loss"), marker="o", ms=4, color="#c2410c", zorder=3)
ax[0].axhline(math.log(2), ls="--", c="gray", lw=1, zorder=1)
ax[0].annotate("ln 2 = no preference", xy=(0.97, math.log(2)),
               xycoords=("axes fraction", "data"), ha="right", va="bottom",
               fontsize=8, color="gray",
               bbox=dict(fc="white", ec="none", alpha=.85, pad=1.5), zorder=4)
ax[0].set_title("train: dpo_loss (down is good)"); ax[0].set_ylabel("loss")

# --- 2. margin + accuracy (train) ---------------------------------------
axa = ax[1].twinx()
axa.plot(*series("accuracy"), marker="s", ms=3, lw=1, alpha=.45,
         color="#15803d", label="accuracy", zorder=1)
axa.axhline(0.5, ls=":", c="#15803d", lw=1, alpha=.6, zorder=1)
axa.set_ylim(0, 1.05); axa.set_ylabel("accuracy", color="#15803d")
axa.tick_params(axis="y", colors="#15803d")
ax[1].plot(*series("margin"), marker="o", ms=4, color="#1d4ed8",
           label="margin (unscaled)", zorder=3)
ax[1].axhline(0, ls="--", c="gray", lw=1, zorder=2)
ax[1].set_ylabel("margin (log units)", color="#1d4ed8")
ax[1].set_zorder(axa.get_zorder() + 1); ax[1].patch.set_visible(False)
ax[1].set_title("train: margin + accuracy")
h1, l1 = ax[1].get_legend_handles_labels(); h2, l2 = axa.get_legend_handles_labels()
ax[1].legend(h1 + h2, l1 + l2, loc="upper left", fontsize=8, framealpha=.9)

# --- 3. implicit rewards: catches likelihood displacement ---------------
ax[2].plot(*series("chosen_reward"), marker="o", ms=4, color="#15803d", label="chosen")
ax[2].plot(*series("rejected_reward"), marker="s", ms=4, color="#b91c1c", label="rejected")
ax[2].axhline(0, ls="--", c="gray", lw=1)
ax[2].legend(fontsize=8, loc="lower left"); ax[2].set_ylabel("implicit reward")
ax[2].set_title("train: implicit rewards\n(watch for BOTH falling)")

# --- 4. THE HEADLINE: held-out, same pairs every time --------------------
# Plotted as a change from the step-0 baseline, so all three start at 0 and the
# question "did chosen go up or did rejected just collapse?" is answerable by eye.
b = eval_history[0]
for key, color, mk, lbl in (("margin", "#1d4ed8", "o", "margin"),
                            ("chosen", "#15803d", "o", "chosen"),
                            ("rejected", "#b91c1c", "s", "rejected")):
    xs, ys = series(key, eval_history)
    dy = [y - b[key] for y in ys]
    ax[3].plot(xs, dy, marker=mk, ms=5, lw=2 if key == "margin" else 1.5,
               color=color, label=lbl, alpha=1.0 if key == "margin" else .8)
    # Label the endpoint: `chosen` is often small next to `rejected` on a shared
    # axis, and its sign is the whole question, so make it readable.
    if dy:
        ax[3].annotate(f"{dy[-1]:+.2f}", xy=(xs[-1], dy[-1]), xytext=(4, 0),
                       textcoords="offset points", va="center", fontsize=8,
                       color=color, fontweight="bold")
ax[3].axhline(0, ls="--", c="gray", lw=1)
ax[3].legend(fontsize=8, loc="upper left")
ax[3].set_ylabel("change from step 0 (nats)")
ax[3].set_title("HELD OUT: change vs baseline\n(margin up + chosen up = real)")

for a in ax:
    a.set_xlabel("optimizer step"); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

## 9. Read the actual text

Save the tuned weights as a sampler snapshot and generate from the frozen reference and the tuned model
on the **same held-out prompts**.

In [ ]:
final_path = training_client.save_weights_for_sampler("dpo-final").result().path
tuned_client = service.create_sampling_client(model_path=final_path, tokenizer=tokenizer)
print("tuned snapshot:", final_path)

# Template markers that must never appear in a decoded answer. If they do, we
# are looking at raw scaffolding rather than content.
_TEMPLATE_MARKERS = ("<|open|>", "<|close|>", "<|sep|>", "<|end_of_msg|>")

# DPO pushes toward longer output (the loss SUMS logprobs, and this dataset's
# chosen responses average 277 tokens against rejected's 185), so the tuned
# model needs more headroom than the base model did. Measured: at 768 the tuned
# model truncated on prompts the base model answered comfortably.
MAX_GEN_TOKENS = 2048


def prompt_messages(row):
    """Everything before the final assistant turn. None if the row isn't that shape."""
    msgs = (normalize_preference_row(row) or {}).get("chosen", {}).get("messages") or []
    if not msgs or msgs[-1].get("role") != "assistant":
        return None
    return msgs[:-1] or None


def generate(client, messages, max_tokens=None, temperature=0.7, timeout=420):
    """Sample one response. Returns (text, status); status "ok" or a skip reason.

    Do NOT gate on parse_response's termination flag: this renderer pre-fills
    `<|open|>response<|sep|>` in the prompt, so every generation -- including
    clean ones -- reports `malformed`. Validate the content instead.
    """
    max_tokens = max_tokens or MAX_GEN_TOKENS
    prompt = renderer.build_generation_prompt(messages)
    params = tinker.SamplingParams(max_tokens=max_tokens, temperature=temperature,
                                   stop=renderer.get_stop_sequences())
    try:
        resp = client.sample(prompt=prompt, num_samples=1,
                             sampling_params=params).result(timeout=timeout)
    except Exception as e:
        # 503 while the pool loads an adapter. Skip one prompt rather than
        # abandoning 80 sequential samples.
        return "", f"sample failed ({type(e).__name__})"

    tokens = list(resp.sequences[0].tokens)
    text = get_text_content(renderer.parse_response(tokens)[0]).strip()
    if not text:
        return "", "empty"
    if len(tokens) >= max_tokens:
        return "", f"truncated at {max_tokens} tokens"
    if any(m in text for m in _TEMPLATE_MARKERS):
        return "", "template markers in output (renderer/prompt mismatch)"
    return text, "ok"


def warm_up(client, label, tries=8, wait=20):
    """Prime a pooled sampler: the pool 503s while loading an adapter on first
    use, which can exhaust the SDK's retries. A 16-token probe is enough -- it
    reports "truncated", which still proves the adapter is serving.
    """
    for i in range(1, tries + 1):
        _, st = generate(client, [{"role": "user", "content": "Say OK."}],
                         max_tokens=16, timeout=120)
        if st == "ok" or st.startswith("truncated"):
            print(f"  {label:24s} ready")
            return True
        print(f"  {label:24s} not ready ({st}) [{i}/{tries}] -- waiting {wait}s")
        if i < tries:
            time.sleep(wait)
    print(f"  {label:24s} STILL UNAVAILABLE -- the pool may be at capacity. Re-run this cell.")
    return False


print("warming up the samplers (the pool loads each adapter on first use):")
ready_base = warm_up(reference_client, "base (reference)")
ready_tuned = warm_up(tuned_client, "tuned")
if not (ready_base and ready_tuned):
    print("\n  One or both samplers are not serving yet. Re-run this cell before section 10;")
    print("  starting the win-rate now would skip every prompt.")
print()

for row in eval_rows[:2]:
    msgs = prompt_messages(row)
    if not msgs:
        continue
    print("=" * 100)
    print("PROMPT:", str(msgs[-1].get("content", ""))[:300])
    for label, client in (("BASE (frozen reference)", reference_client), ("AFTER DPO", tuned_client)):
        text, status = generate(client, msgs)
        print(f"\n--- {label} ---")
        print(f" {text[:700]}" if status == "ok" else f" (unusable: {status})")
    print()

## 10. Win-rate: the number you show a customer

Margin is the honest training signal, but it isn't persuasive on a slide. So we ask an **impartial judge
model** to compare base and tuned answers on the same held-out prompts and report how often the tuned
model wins.

Four things make this a fair test rather than a flattering one:

- **A different model family judges** (`glm-5p2` scoring `kimi-k3`), so there's no self-preference bias.
- **`swap_guard=True`** runs both orderings and keeps a verdict only when they agree after un-swapping,
  which removes position bias. Disagreements become ties.
- **Held-out prompts only** — never trained on.
- **And a caveat stated before you see the chart:** 12 prompts with a swap guard typically leaves only a
  handful of *decisive* verdicts. Treat this as a smoke test of the measurement, not an effect size. The
  number you'd publish comes from 200+ prompts.

No deployment needed: we generate from the two snapshots we already have and only call the API for the
judge. Note `pairwise_judge` swallows judge errors as `"tie"`, so an all-ties result usually means a bad
`JUDGE_MODEL` id rather than a tied model.

In [ ]:
from eval_common import fw_client, pairwise_judge


def _norm_verdict(v):
    """Collapse a judge verdict to exactly "A", "B", or "tie"."""
    v = str(v).strip().upper()
    return v if v in ("A", "B") else "tie"


def judge_pair(judge, model, messages, a, b):
    """Swap-guarded pairwise verdict: "A", "B", or "tie".

    Both orderings; a decisive verdict only when they agree after un-swapping,
    so position bias becomes a tie. Same two judge calls as swap_guard=True --
    but the shared helper's remap KeyErrors when a judge returns "Tie" (it
    compares lowercase then applies .upper()), so we normalize and un-swap here.
    """
    v1 = _norm_verdict(pairwise_judge(judge, model, messages, a, b, swap_guard=False))
    v2 = _norm_verdict(pairwise_judge(judge, model, messages, b, a, swap_guard=False))
    v2 = {"A": "B", "B": "A", "tie": "tie"}[v2]      # un-swap: "A" there means b won
    return v1 if v1 == v2 else "tie"

judge = fw_client(API_KEY, base_url=f"{API_BASE.rstrip('/')}/inference/v1")
subjects = [r for r in eval_rows if prompt_messages(r)][:N_WINRATE]

# Both adapters must be serving before 80 sequential samples. warm_up() is
# defined in section 9 -- run that cell first.
assert warm_up(reference_client, "base (reference)") and warm_up(tuned_client, "tuned"), (
    "samplers are not serving; re-run this cell (the pool loads adapters on demand)"
)
print()

wins = losses = ties = 0
# Count skips PER SIDE. If one model is skipped more than the other, the
# surviving sample is biased -- e.g. dropping the tuned model's long answers
# would silently measure only the prompts where it happened to be concise.
skips = {"base": {}, "tuned": {}}

for n, row in enumerate(subjects, 1):
    msgs = prompt_messages(row)
    base_ans, base_st = generate(reference_client, msgs)
    tuned_ans, tuned_st = generate(tuned_client, msgs)
    if base_st != "ok":
        skips["base"][base_st] = skips["base"].get(base_st, 0) + 1
    if tuned_st != "ok":
        skips["tuned"][tuned_st] = skips["tuned"].get(tuned_st, 0) + 1
    if base_st != "ok" or tuned_st != "ok":
        print(f"  [{n}/{len(subjects)}] skipped  (base: {base_st} | tuned: {tuned_st})")
        continue

    # A = base, B = tuned  ->  "B" means the tuned model won.
    # Defensive: this is shared code and the judge is a live model. A malformed
    # verdict should cost one prompt, not the whole 80-sample run.
    try:
        verdict = judge_pair(judge, JUDGE_MODEL, msgs, base_ans, tuned_ans)
    except Exception as e:
        print(f"  [{n}/{len(subjects)}] judge error ({type(e).__name__}) -- counting as tie")
        verdict = "tie"
    wins += verdict == "B"
    losses += verdict == "A"
    ties += verdict == "tie"
    print(f"  [{n}/{len(subjects)}] {'TUNED' if verdict == 'B' else 'BASE' if verdict == 'A' else 'tie'}")

decisive = wins + losses
win_rate = wins / decisive if decisive else 0.0
skipped = sum(sum(v.values()) for v in skips.values()) and len(subjects) - (wins + losses + ties)

print(f"\ntuned won {wins} / lost {losses} / tied {ties} / skipped {skipped}")
for side in ("base", "tuned"):
    if skips[side]:
        print(f"  {side:5s} skips: " + ", ".join(f"{k} x{v}" for k, v in skips[side].items()))
n_base, n_tuned = sum(skips['base'].values()), sum(skips['tuned'].values())
if abs(n_base - n_tuned) > max(2, 0.1 * len(subjects)):
    print(f"  !! ASYMMETRIC SKIPS ({n_base} base vs {n_tuned} tuned) -- the surviving")
    print(f"     sample is biased toward prompts the skipped side handled well.")
    print(f"     Raise MAX_GEN_TOKENS and re-run before trusting this number.")
print(f"\nwin-rate vs base over {decisive} decisive verdicts: {win_rate:.1%}"
      if decisive else "\nno decisive verdicts -- check JUDGE_MODEL")

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
bars = ax.bar(["base\n(frozen reference)", "after DPO"], [1 - win_rate, win_rate],
              color=["#94a3b8", "#15803d"], width=.6)
for b, v in zip(bars, [1 - win_rate, win_rate]):
    ax.text(b.get_x() + b.get_width() / 2, v + .02, f"{v:.0%}", ha="center", fontweight="bold")
ax.axhline(.5, ls="--", c="gray", lw=1)
ax.annotate("50% = no difference", (.02, .51), xycoords=("axes fraction", "data"),
            fontsize=8, color="gray")
ax.set_ylim(0, 1.1); ax.set_ylabel("share of decisive judgments")
ax.set_title(f"Head-to-head, judged by {JUDGE_MODEL.split('/')[-1]}\n"
             f"{wins}-{losses} over {decisive} decisive verdicts "
             f"({ties} ties, {skipped} skipped, {len(subjects)} prompts)")
ax.grid(axis="y", alpha=.3); plt.tight_layout(); plt.show()

## 11. Ship it (optional)

Promotion copies a checkpoint **out of** this session into the model catalog as
`accounts/<account>/models/<output_model_id>`.

**The promotion *call* is session-scoped; the model it produces is not.** The sampler checkpoint it
reads from dies with the session, so this cell has to run now — but what you get is durable.

**A promoted model is not running.** It is a registered LoRA adapter, no GPUs and no endpoint. Serving
it is a separate step (`firectl deployment create <model>`), which bills per replica-hour.

```
session snapshot  ->  usable now, this kernel only   (sections 9-10 sample this)
save_state ref    ->  portable, resumable            (section 5 writes these)
promoted model    ->  durable, nothing running        (this cell)
deployment        ->  live endpoint, billed hourly    (not in this notebook)
```

In [ ]:
PROMOTE = False              # True -> register a durable model (does NOT deploy it)
OUTPUT_MODEL_ID = "dpo-ultrafeedback-demo"

if PROMOTE:
    from fireworks.training.sdk import FireworksClient

    def control_plane_base_url(base_url: str) -> str:
        root = base_url.rstrip("/")
        suffix = "/training/v1/serverless"
        return root[: -len(suffix)] if root.endswith(suffix) else root

    cp = FireworksClient(api_key=API_KEY, base_url=control_plane_base_url(API_BASE))
    try:
        # The training SESSION id lives on the service (a "ts-..." id), NOT on
        # training_client.session_id -- that one only disambiguates snapshot names.
        session = (service.training_session_name
                   or f"accounts/{cp.account_id}/trainingSessions/{service.training_session_id}")
        candidates = [c for c in cp.list_training_session_checkpoints(session) if c.get("promotable")]
        print("promotable checkpoints:", [c.get("name") for c in candidates])
        match = next((c for c in candidates if "dpo-final" in str(c.get("name", ""))), None)
        if match:
            model = cp.promote_session_checkpoint(
                name=match["name"], output_model_id=OUTPUT_MODEL_ID, base_model=BASE_MODEL)
            print("promoted:", model)
        else:
            print("dpo-final is not listed as promotable yet")
    finally:
        cp.close()
else:
    print("skipped. set PROMOTE = True to register the checkpoint as a durable model.")

## 12. Cleanup

Serverless has no deployment to delete. This is the entire teardown story.

In [ ]:
for c in (reference_client, tuned_client):
    try:
        c.close()
    except Exception:
        pass
print("closed sampling clients. nothing left running.")